SQL Interview Questions by Level

# 🟢 JUNIOR SQL QUESTIONS 



## 1. Basic Queries
### Q1.1: Explain SELECT, WHERE, ORDER BY, and LIMIT with examples

解释 SELECT, WHERE, ORDER BY, LIMIT 的用法

> "SELECT specifies which columns to retrieve — use * for all columns or list specific ones. WHERE filters rows based on conditions before any processing. ORDER BY sorts results — ASC for ascending (default), DESC for descending. LIMIT restricts how many rows are returned, essential for pagination. The execution order is: FROM gets the table, WHERE filters rows, SELECT picks columns, ORDER BY sorts, then LIMIT caps the output. Always filter with WHERE before sorting to improve performance."


**四个关键字的作用：**

| 关键字 | 作用 | 执行顺序 |
|--------|------|----------|
| SELECT | 选择要返回的列 | 第 3 |
| WHERE | 过滤行（条件筛选） | 第 2 |
| ORDER BY | 排序结果 | 第 4 |
| LIMIT | 限制返回行数 | 第 5 |
| FROM | 指定数据来源表 | 第 1 |

**代码示例：**

```sql
-- 基础查询：选择特定列
SELECT name, email, created_at
FROM users
WHERE status = 'active'
ORDER BY created_at DESC
LIMIT 10;

-- 多条件过滤
SELECT * FROM orders
WHERE total_amount > 100 
  AND order_date >= '2024-01-01'
  AND status IN ('completed', 'shipped')
ORDER BY total_amount DESC
LIMIT 20 OFFSET 10;  -- 分页：跳过前10条，取20条
```

**常见陷阱：**
- `SELECT *` 在生产环境中应避免（浪费带宽、无法利用覆盖索引）
- `ORDER BY` 没有索引支持时会很慢
- `LIMIT` 不加 `ORDER BY` 时结果顺序不确定


### Q1.2: What's the difference between DISTINCT and GROUP BY for removing duplicates?

去重时 DISTINCT 和 GROUP BY 有什么区别？

> "Both can remove duplicates, but they serve different purposes. DISTINCT simply removes duplicate rows from the result — it's straightforward and optimized for this use case. GROUP BY is designed for aggregation — it groups rows and lets you apply aggregate functions like COUNT or SUM. For pure deduplication, DISTINCT is cleaner and often faster. However, if you need aggregates or want to see how many duplicates exist, use GROUP BY. Some databases optimize them identically, but semantically DISTINCT signals 'I want unique rows' while GROUP BY signals 'I want to aggregate.'"


**核心区别：**

| 特性 | DISTINCT | GROUP BY |
|------|----------|----------|
| 主要用途 | 纯去重 | 分组聚合 |
| 可用聚合函数 | ❌ | ✅ COUNT, SUM 等 |
| 语义清晰度 | 更清晰表示"去重" | 表示"分组" |
| 性能 | 通常相同或更快 | 取决于数据库优化器 |

**代码对比：**

```sql
-- 需求：获取所有不重复的部门名称

-- 方法1：DISTINCT（推荐，语义清晰）
SELECT DISTINCT department FROM employees;

-- 方法2：GROUP BY（功能相同，但语义不如 DISTINCT 清晰）
SELECT department FROM employees GROUP BY department;

-- 需求：获取每个部门的人数（必须用 GROUP BY）
SELECT department, COUNT(*) as emp_count
FROM employees
GROUP BY department;

-- 需求：多列去重
SELECT DISTINCT first_name, last_name FROM employees;
-- 等价于
SELECT first_name, last_name FROM employees GROUP BY first_name, last_name;
```

**选择建议：**
- 纯去重 → 用 `DISTINCT`
- 需要聚合函数 → 用 `GROUP BY`
- 需要知道重复次数 → 用 `GROUP BY` + `COUNT(*)`



### Q1.3: How do you handle NULL values in SQL?

SQL 中如何处理 NULL 值？

> "NULL represents unknown or missing data, not zero or empty string. You can't compare NULL with equals — use IS NULL or IS NOT NULL instead. In calculations, NULL propagates — any arithmetic with NULL returns NULL. For sorting, NULLs typically appear first in ASC or last in DESC, but this varies by database. Use COALESCE to provide default values — it returns the first non-NULL argument. NULLIF does the reverse — returns NULL if two values are equal. In aggregations, most functions like SUM and AVG ignore NULLs, but COUNT(*) counts all rows while COUNT(column) excludes NULLs."


**NULL 的特殊性：**
- NULL ≠ 0
- NULL ≠ 空字符串 ''
- NULL = NULL 的结果是 **UNKNOWN**（不是 TRUE）

**NULL 比较规则：**

```sql
-- ❌ 错误：不能用 = 比较 NULL
SELECT * FROM users WHERE phone = NULL;     -- 永远返回空！
SELECT * FROM users WHERE phone <> NULL;    -- 永远返回空！

-- ✅ 正确：用 IS NULL / IS NOT NULL
SELECT * FROM users WHERE phone IS NULL;
SELECT * FROM users WHERE phone IS NOT NULL;
```

**NULL 处理函数：**

```sql
-- COALESCE：返回第一个非 NULL 值（最常用）
SELECT COALESCE(nickname, name, 'Anonymous') AS display_name FROM users;

-- IFNULL (MySQL) / ISNULL (SQL Server)：两参数版本
SELECT IFNULL(phone, 'N/A') FROM users;  -- MySQL
SELECT ISNULL(phone, 'N/A') FROM users;  -- SQL Server

-- NULLIF：相等时返回 NULL（避免除零错误）
SELECT total / NULLIF(count, 0) AS average FROM stats;  -- count=0 时返回 NULL 而非报错

-- NVL (Oracle)：类似 COALESCE
SELECT NVL(phone, 'N/A') FROM users;
```

**聚合函数与 NULL：**

```sql
-- 假设 salary 列有值：100, 200, NULL, 300
SELECT COUNT(*)        FROM employees;  -- 4（计算所有行）
SELECT COUNT(salary)   FROM employees;  -- 3（排除 NULL）
SELECT SUM(salary)     FROM employees;  -- 600（忽略 NULL）
SELECT AVG(salary)     FROM employees;  -- 200（600/3，不是 600/4）
```

**排序中的 NULL：**

```sql
-- 不同数据库默认行为不同
-- PostgreSQL: NULLS LAST (ASC) / NULLS FIRST (DESC) - 可显式指定
SELECT * FROM users ORDER BY phone ASC NULLS LAST;
SELECT * FROM users ORDER BY phone DESC NULLS FIRST;

-- MySQL: NULL 被视为最小值
-- SQL Server: NULL 被视为最小值
```


### Q1.4: How do you use LIKE and wildcards for pattern matching?

如何使用 LIKE 和通配符进行模式匹配？

> "LIKE is used for pattern matching in WHERE clauses. There are two wildcards: percent sign matches zero or more characters, underscore matches exactly one character. For example, 'A%' matches anything starting with A, '%son' matches anything ending with son, '%john%' matches anything containing john, and 'A_B' matches exactly 3 characters starting with A and ending with B. Be cautious with leading wildcards like '%john' — they prevent index usage and cause full table scans. If you need case-insensitive matching, use ILIKE in PostgreSQL or LOWER() with LIKE in other databases. For complex patterns, consider regular expressions with REGEXP."

**两个通配符：**

| 通配符 | 含义 | 示例 |
|--------|------|------|
| `%` | 匹配 0 个或多个任意字符 | `'A%'` 匹配 "A", "AB", "Apple" |
| `_` | 匹配恰好 1 个任意字符 | `'A_'` 匹配 "AB", "AC"，不匹配 "A" 或 "ABC" |

**常用模式：**

```sql
-- 以...开头（可以用索引 ✅）
SELECT * FROM users WHERE name LIKE 'John%';

-- 以...结尾（不能用索引 ❌）
SELECT * FROM users WHERE name LIKE '%son';

-- 包含...（不能用索引 ❌）
SELECT * FROM users WHERE name LIKE '%john%';

-- 固定长度模式
SELECT * FROM products WHERE code LIKE 'PRD-____';  -- PRD- 后跟恰好4个字符

-- 组合使用
SELECT * FROM users WHERE email LIKE '%@gmail.com';
```

**大小写处理：**

```sql
-- PostgreSQL: ILIKE（不区分大小写）
SELECT * FROM users WHERE name ILIKE 'john%';

-- 其他数据库：转换为统一大小写
SELECT * FROM users WHERE LOWER(name) LIKE 'john%';
SELECT * FROM users WHERE UPPER(name) LIKE 'JOHN%';
```

**转义特殊字符：**

```sql
-- 如果要匹配 % 或 _ 本身
SELECT * FROM products WHERE name LIKE '%50\%%' ESCAPE '\';  -- 匹配包含 "50%" 的
SELECT * FROM files WHERE name LIKE 'file\_v1%' ESCAPE '\';  -- 匹配 "file_v1" 开头的
```

**性能警告：**

| 模式 | 能用索引？ | 性能 |
|------|-----------|------|
| `'abc%'` | ✅ 能 | 快 |
| `'%abc'` | ❌ 不能 | 慢（全表扫描） |
| `'%abc%'` | ❌ 不能 | 慢（全表扫描） |
| `'_abc'` | ❌ 不能 | 慢 |

**替代方案：**
- 全文搜索 → Full-Text Search (FTS)
- 复杂模式 → `REGEXP` / `SIMILAR TO`
- 前缀匹配大量数据 → 创建专用索引



### Q1.5: What are common date functions and how do you use them?

常用的日期函数有哪些？如何使用？

> "Date functions vary by database but common operations include: getting current date/time with CURRENT_DATE, NOW(), or GETDATE(); extracting parts with EXTRACT(), YEAR(), MONTH(), DAY(); formatting with TO_CHAR() or DATE_FORMAT(); date arithmetic with DATE_ADD(), DATE_SUB(), or interval syntax; and truncating with DATE_TRUNC(). For filtering date ranges, always use explicit comparisons rather than functions on columns to preserve index usage. When comparing dates, be careful with timestamps — use date ranges or DATE() to strip time components. Different databases have different syntax, so always check the documentation."


**获取当前日期/时间：**

| 功能 | PostgreSQL | MySQL | SQL Server |
|------|------------|-------|------------|
| 当前日期 | `CURRENT_DATE` | `CURDATE()` | `CAST(GETDATE() AS DATE)` |
| 当前时间戳 | `NOW()` | `NOW()` | `GETDATE()` |
| 当前时间 | `CURRENT_TIME` | `CURTIME()` | `CAST(GETDATE() AS TIME)` |

**提取日期部分：**

```sql
-- PostgreSQL / 标准 SQL
SELECT EXTRACT(YEAR FROM order_date) AS year,
       EXTRACT(MONTH FROM order_date) AS month,
       EXTRACT(DAY FROM order_date) AS day
FROM orders;

-- MySQL
SELECT YEAR(order_date), MONTH(order_date), DAY(order_date) FROM orders;

-- 提取星期几
SELECT EXTRACT(DOW FROM order_date) FROM orders;  -- PostgreSQL: 0=周日
SELECT DAYOFWEEK(order_date) FROM orders;         -- MySQL: 1=周日
```

**日期格式化：**

```sql
-- PostgreSQL
SELECT TO_CHAR(order_date, 'YYYY-MM-DD') FROM orders;
SELECT TO_CHAR(order_date, 'Mon DD, YYYY') FROM orders;  -- Jan 15, 2024

-- MySQL
SELECT DATE_FORMAT(order_date, '%Y-%m-%d') FROM orders;
SELECT DATE_FORMAT(order_date, '%b %d, %Y') FROM orders;
```

**日期运算：**

```sql
-- PostgreSQL: 使用 INTERVAL
SELECT order_date + INTERVAL '7 days' FROM orders;
SELECT order_date - INTERVAL '1 month' FROM orders;
SELECT NOW() - created_at AS age FROM users;  -- 返回 INTERVAL

-- MySQL
SELECT DATE_ADD(order_date, INTERVAL 7 DAY) FROM orders;
SELECT DATE_SUB(order_date, INTERVAL 1 MONTH) FROM orders;
SELECT DATEDIFF(NOW(), created_at) AS days_old FROM users;  -- 返回天数

-- 日期差值
SELECT order_date - created_at FROM orders;  -- PostgreSQL: 返回天数
SELECT DATEDIFF(order_date, created_at) FROM orders;  -- MySQL
```

**日期截断（对于分组很有用）：**

```sql
-- PostgreSQL
SELECT DATE_TRUNC('month', order_date) AS month, SUM(amount)
FROM orders
GROUP BY DATE_TRUNC('month', order_date);

-- MySQL
SELECT DATE_FORMAT(order_date, '%Y-%m-01') AS month, SUM(amount)
FROM orders
GROUP BY DATE_FORMAT(order_date, '%Y-%m-01');
```

**日期范围过滤（性能最佳写法）：**

```sql
-- ✅ 推荐：直接比较（可用索引）
SELECT * FROM orders 
WHERE order_date >= '2024-01-01' 
  AND order_date < '2024-02-01';

-- ❌ 避免：在列上使用函数（无法用索引）
SELECT * FROM orders 
WHERE YEAR(order_date) = 2024 AND MONTH(order_date) = 1;
```

---



## 2. JOIN Operations



### Q2.1: Explain INNER JOIN vs LEFT JOIN vs RIGHT JOIN

解释 INNER JOIN、LEFT JOIN、RIGHT JOIN 的区别


> "INNER JOIN returns only rows that have matching values in both tables — if there's no match, the row is excluded. LEFT JOIN returns all rows from the left table and matching rows from the right — when there's no match, NULL values fill the right table's columns. RIGHT JOIN is the mirror — all rows from the right table with NULLs for non-matching left rows. In practice, LEFT JOIN is most common; RIGHT JOIN can always be rewritten as LEFT JOIN by swapping table order. A key pattern is LEFT JOIN with WHERE right.id IS NULL to find rows in the left table that have no match in the right."


**三种 JOIN 图解：**

```text
表 A (users)          表 B (orders)
┌────┬───────┐       ┌────┬─────────┐
│ id │ name  │       │ id │ user_id │
├────┼───────┤       ├────┼─────────┤
│ 1  │ Alice │       │ 1  │ 1       │
│ 2  │ Bob   │       │ 2  │ 1       │
│ 3  │ Carol │       │ 3  │ 3       │
└────┴───────┘       └────┴─────────┘

INNER JOIN: 只返回 Alice(2条), Carol(1条) — Bob 没有订单被排除
LEFT JOIN:  返回 Alice(2条), Bob(1条,order为NULL), Carol(1条)
RIGHT JOIN: 返回 Alice(2条), Carol(1条) — 所有订单都有对应用户
```

**代码示例：**

```sql
-- INNER JOIN: 只返回有订单的用户
SELECT u.name, o.id AS order_id
FROM users u
INNER JOIN orders o ON u.id = o.user_id;
-- 结果: Alice-1, Alice-2, Carol-3

-- LEFT JOIN: 所有用户，没有订单的显示 NULL
SELECT u.name, o.id AS order_id
FROM users u
LEFT JOIN orders o ON u.id = o.user_id;
-- 结果: Alice-1, Alice-2, Bob-NULL, Carol-3

-- RIGHT JOIN: 所有订单，没有用户的显示 NULL
SELECT u.name, o.id AS order_id
FROM users u
RIGHT JOIN orders o ON u.id = o.user_id;
-- 通常改写为 LEFT JOIN（交换表顺序）
```

**常用模式：找出没有匹配的记录**

```sql
-- 找出没有下过订单的用户
SELECT u.*
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
WHERE o.id IS NULL;

-- 等价写法（使用 NOT EXISTS）
SELECT * FROM users u
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id);

-- 等价写法（使用 NOT IN，注意 NULL 问题）
SELECT * FROM users
WHERE id NOT IN (SELECT user_id FROM orders WHERE user_id IS NOT NULL);
```

**性能提示：**
- JOIN 列应该有索引
- 小表放在 JOIN 的右边（或让优化器决定）
- 先 WHERE 过滤再 JOIN 可以减少数据量


### Q2.2: How do you write a multi-table JOIN?

如何编写多表 JOIN？

> "For multi-table joins, chain them sequentially — each JOIN adds one more table to the result. Order matters for readability but optimizers usually reorder for performance. Start with your primary table, then join related tables one by one. Use table aliases for clarity. Be careful with join types — mixing INNER and LEFT joins affects which rows survive. A common pattern is joining a fact table with multiple dimension tables in a star schema. Always specify join conditions explicitly with ON clauses rather than comma-separated tables with WHERE conditions for better readability and to avoid accidental Cartesian products."


**多表 JOIN 语法：**

```sql
-- 基本结构：链式 JOIN
SELECT ...
FROM table_a a
JOIN table_b b ON a.id = b.a_id
JOIN table_c c ON b.id = c.b_id
JOIN table_d d ON a.id = d.a_id;
```

**实际示例：电商订单查询**

```sql
-- 查询订单详情：订单 + 用户 + 商品 + 商品分类
SELECT 
    o.id AS order_id,
    o.order_date,
    u.name AS customer_name,
    u.email,
    p.name AS product_name,
    c.name AS category_name,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS subtotal
FROM orders o
INNER JOIN users u ON o.user_id = u.id
INNER JOIN order_items oi ON o.id = oi.order_id
INNER JOIN products p ON oi.product_id = p.id
LEFT JOIN categories c ON p.category_id = c.id  -- 商品可能没有分类
WHERE o.order_date >= '2024-01-01'
ORDER BY o.order_date DESC;
```

**JOIN 类型混用的影响：**

```sql
-- 场景：有些用户没有订单，有些订单没有商品
-- 如果要保留所有用户：
SELECT u.name, o.id, oi.product_id
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
LEFT JOIN order_items oi ON o.id = oi.order_id;  -- 也必须是 LEFT JOIN

-- ❌ 错误：第二个 INNER JOIN 会过滤掉没有订单的用户
SELECT u.name, o.id, oi.product_id
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
INNER JOIN order_items oi ON o.id = oi.order_id;  -- Bob 被过滤掉了
```

**多表 JOIN 最佳实践：**

| 建议 | 原因 |
|------|------|
| 使用有意义的表别名 | `u` for users, `o` for orders |
| 显式写 JOIN 类型 | 不要省略 INNER |
| 每个 JOIN 单独一行 | 便于阅读和调试 |
| 先过滤后 JOIN | 减少 JOIN 的数据量 |
| 检查 JOIN 条件 | 缺少条件会产生笛卡尔积 |


### Q2.3: What is a self join and when would you use it?

什么是自连接？什么时候使用？


> "A self join is when a table is joined with itself — you use different aliases to treat it as two separate tables. Common use cases include: hierarchical data like employee-manager relationships where both are in the same employees table; finding pairs or combinations within a dataset; comparing rows within the same table like finding customers in the same city; and time-series comparisons like comparing today's value with yesterday's. The key is using clear aliases to distinguish the two 'copies' of the table and ensuring your join condition doesn't create duplicates or include self-matches where inappropriate."

**什么是自连接？**
- 表与自己进行 JOIN
- 必须使用不同的别名区分"两个表"

**场景 1：层级关系（员工-经理）**

```sql
-- 表结构: employees (id, name, manager_id)
-- manager_id 指向同一张表的 id

-- 查询每个员工及其经理的名字
SELECT 
    e.name AS employee,
    m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.id;

-- 结果示例:
-- | employee | manager |
-- | Alice    | Bob     |
-- | Bob      | Carol   |
-- | Carol    | NULL    |  -- Carol 是最高层，没有经理
```

**场景 2：找出同一城市的客户对**

```sql
-- 找出住在同一城市的不同客户
SELECT 
    a.name AS customer1,
    b.name AS customer2,
    a.city
FROM customers a
INNER JOIN customers b ON a.city = b.city AND a.id < b.id;  -- id < 避免重复配对

-- a.id < b.id 的作用:
-- ✅ 返回: (Alice, Bob, NYC)
-- ❌ 避免: (Bob, Alice, NYC) -- 重复
-- ❌ 避免: (Alice, Alice, NYC) -- 自己配自己
```

**场景 3：找出连续日期的数据**

```sql
-- 找出连续两天都有登录的用户
SELECT DISTINCT a.user_id
FROM logins a
INNER JOIN logins b ON a.user_id = b.user_id 
                   AND a.login_date = b.login_date + INTERVAL '1 day';
```

**场景 4：比较前后记录（替代 LAG/LEAD）**

```sql
-- 旧数据库没有窗口函数时，用自连接模拟 LAG
SELECT 
    curr.date,
    curr.price,
    prev.price AS prev_price,
    curr.price - prev.price AS change
FROM stock_prices curr
LEFT JOIN stock_prices prev ON curr.date = prev.date + INTERVAL '1 day';
```

**自连接注意事项：**
- 必须用别名区分两个"表"
- 注意避免重复配对（用 `a.id < b.id` 或 `a.id != b.id`）
- 大表自连接性能较差，考虑用窗口函数替代

---



## 3. Aggregate Functions 

### Q3.1: Explain COUNT, SUM, AVG, MAX, MIN and their differences

解释 COUNT, SUM, AVG, MAX, MIN 的区别

> "These are aggregate functions that collapse multiple rows into a single value. COUNT counts rows — COUNT(*) counts all rows including NULLs, while COUNT(column) only counts non-NULL values. SUM adds up numeric values, ignoring NULLs. AVG calculates the mean, also ignoring NULLs — so the divisor is the count of non-NULL values, not total rows. MAX and MIN find the highest and lowest values respectively, working with numbers, strings, and dates. All these functions except COUNT(*) ignore NULL values. A common mistake is using AVG when you actually want SUM divided by total row count — these differ when NULLs exist."

**五个聚合函数：**

| 函数 | 作用 | NULL 处理 |
|------|------|-----------|
| COUNT(*) | 计算所有行数 | 包含 NULL 行 |
| COUNT(列) | 计算非 NULL 值的数量 | 排除 NULL |
| SUM(列) | 求和 | 忽略 NULL |
| AVG(列) | 求平均值 | 忽略 NULL（分母是非 NULL 数量） |
| MAX(列) | 最大值 | 忽略 NULL |
| MIN(列) | 最小值 | 忽略 NULL |

**NULL 处理示例：**

```sql
-- 假设数据: [100, 200, NULL, 300]
SELECT 
    COUNT(*)        AS total_rows,      -- 4
    COUNT(salary)   AS non_null_count,  -- 3
    SUM(salary)     AS total_salary,    -- 600
    AVG(salary)     AS avg_salary,      -- 200 (600/3, 不是 600/4)
    MAX(salary)     AS max_salary,      -- 300
    MIN(salary)     AS min_salary       -- 100
FROM employees;
```

**COUNT 的三种用法：**

```sql
-- COUNT(*): 所有行（最快，不需要读取具体列）
SELECT COUNT(*) FROM orders;

-- COUNT(列): 非 NULL 值数量
SELECT COUNT(phone) FROM users;  -- 有多少用户填了电话

-- COUNT(DISTINCT 列): 不重复值数量
SELECT COUNT(DISTINCT user_id) FROM orders;  -- 有多少不同的用户下过单
```

**AVG 的陷阱：**

```sql
-- 场景：计算平均分数，有些学生没参加考试（NULL）
-- 数据: [80, 90, NULL, 70]

-- AVG 会忽略 NULL
SELECT AVG(score) FROM students;  -- 80 (240/3)

-- 如果想把缺考算作 0 分
SELECT AVG(COALESCE(score, 0)) FROM students;  -- 60 (240/4)

-- 或者
SELECT SUM(score) / COUNT(*) FROM students;  -- 60
```

**条件聚合：**

```sql
-- 在一个查询中计算多个条件的统计
SELECT 
    COUNT(*) AS total_orders,
    COUNT(CASE WHEN status = 'completed' THEN 1 END) AS completed_orders,
    SUM(CASE WHEN status = 'completed' THEN amount ELSE 0 END) AS completed_revenue,
    AVG(CASE WHEN status = 'completed' THEN amount END) AS avg_completed_amount
FROM orders;

-- PostgreSQL/MySQL 简写
SELECT 
    COUNT(*) FILTER (WHERE status = 'completed') AS completed_orders  -- PostgreSQL
FROM orders;
```


### Q3.2: What's the difference between WHERE and HAVING with GROUP BY?

GROUP BY 时 WHERE 和 HAVING 有什么区别？


> "WHERE filters individual rows before grouping, HAVING filters groups after aggregation. WHERE cannot use aggregate functions because aggregates don't exist until after grouping. HAVING can only use aggregate functions or columns in the GROUP BY clause. For performance, always put conditions in WHERE when possible — it reduces the number of rows that need to be grouped. Use HAVING only for conditions on aggregated results. A common pattern is WHERE for row-level filters, GROUP BY to aggregate, then HAVING to filter the aggregated results."


**执行顺序决定了区别：**

```text
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY
        ↑                    ↑
      先过滤行            后过滤组
```

**对比表：**

| 特性 | WHERE | HAVING |
|------|-------|--------|
| 执行时机 | GROUP BY 之前 | GROUP BY 之后 |
| 过滤对象 | 原始行 | 聚合后的组 |
| 能用聚合函数？ | ❌ | ✅ |
| 性能 | 更好（减少分组数据量） | 已经分组完才过滤 |

**代码示例：**

```sql
-- ❌ 错误：WHERE 中不能用聚合函数
SELECT department, COUNT(*) AS emp_count
FROM employees
WHERE COUNT(*) > 5  -- 报错！
GROUP BY department;

-- ✅ 正确：HAVING 过滤聚合结果
SELECT department, COUNT(*) AS emp_count
FROM employees
GROUP BY department
HAVING COUNT(*) > 5;

-- ✅ 最佳：WHERE 和 HAVING 配合
SELECT department, COUNT(*) AS emp_count
FROM employees
WHERE status = 'active'      -- 先过滤：只看在职员工（减少分组数据量）
GROUP BY department
HAVING COUNT(*) > 5;         -- 再过滤：只要人数>5的部门
```

**常见面试陷阱：**

```sql
-- 问：这个查询能优化吗？
SELECT department, AVG(salary)
FROM employees
GROUP BY department
HAVING department = 'Engineering';

-- 答：HAVING 的条件应该移到 WHERE（不是聚合条件）
SELECT department, AVG(salary)
FROM employees
WHERE department = 'Engineering'  -- 更早过滤，性能更好
GROUP BY department;
```

**记忆口诀：**
- WHERE 过滤行，HAVING 过滤组
- 能用 WHERE 就不用 HAVING



### Q3.3: How do you find duplicate records in a table?

如何找出表中的重复记录？


> "There are several approaches. The simplest uses GROUP BY with HAVING COUNT greater than 1 to identify which values are duplicated. To see all duplicate rows with details, use a window function — COUNT OVER PARTITION BY the duplicate columns, then filter where count is greater than 1. To delete duplicates while keeping one, use ROW_NUMBER partitioned by duplicate columns, ordered by some criteria like ID or timestamp, then delete where row_number is greater than 1. For large tables, the window function approach is usually most efficient as it requires only one table scan."

**方法 1：GROUP BY + HAVING（只看哪些值重复）**

```sql
-- 找出重复的 email
SELECT email, COUNT(*) AS duplicate_count
FROM users
GROUP BY email
HAVING COUNT(*) > 1;

-- 结果：只显示值和数量，不显示具体行
-- | email           | duplicate_count |
-- | test@test.com   | 3               |
```

**方法 2：窗口函数（显示所有重复行的详细信息）**

```sql
-- 显示每一条重复记录
WITH duplicates AS (
    SELECT *,
        COUNT(*) OVER (PARTITION BY email) AS dup_count
    FROM users
)
SELECT * FROM duplicates WHERE dup_count > 1;

-- 结果：显示完整的行信息
-- | id | email           | name  | dup_count |
-- | 1  | test@test.com   | Alice | 3         |
-- | 5  | test@test.com   | Bob   | 3         |
-- | 9  | test@test.com   | Carol | 3         |
```

**方法 3：删除重复，保留一条**

```sql
-- 用 ROW_NUMBER 标记，保留 id 最小的
WITH ranked AS (
    SELECT id,
        ROW_NUMBER() OVER (
            PARTITION BY email 
            ORDER BY id  -- 保留最小 id
        ) AS rn
    FROM users
)
DELETE FROM users 
WHERE id IN (SELECT id FROM ranked WHERE rn > 1);

-- 或者保留最新的
WITH ranked AS (
    SELECT id,
        ROW_NUMBER() OVER (
            PARTITION BY email 
            ORDER BY created_at DESC  -- 保留最新
        ) AS rn
    FROM users
)
DELETE FROM users 
WHERE id IN (SELECT id FROM ranked WHERE rn > 1);
```

**多列判断重复：**

```sql
-- 按多列组合判断重复
SELECT first_name, last_name, email, COUNT(*)
FROM users
GROUP BY first_name, last_name, email
HAVING COUNT(*) > 1;
```

**方法对比：**

| 方法 | 优点 | 缺点 |
|------|------|------|
| GROUP BY + HAVING | 简单，只看重复值 | 看不到完整行信息 |
| 窗口函数 | 一次扫描，显示完整信息 | 语法稍复杂 |
| 自连接 | 兼容老数据库 | 大表性能差 |

---



# 🟡 MID-LEVEL SQL QUESTIONS


## 4. Window Functions 
### Q4.1: Explain ROW_NUMBER vs RANK vs DENSE_RANK

ROW_NUMBER、RANK、DENSE_RANK 的区别


> "All three are ranking functions but handle ties differently. ROW_NUMBER assigns unique sequential integers regardless of ties — even equal values get different numbers, making it non-deterministic for ties. RANK assigns the same rank to ties but then skips — so you might get 1, 2, 2, 4 where position 3 is skipped. DENSE_RANK also handles ties but doesn't skip — you'd get 1, 2, 2, 3. Use ROW_NUMBER when you need exactly N rows regardless of ties, like pagination or deduplication. Use RANK when the gap matters, like competition rankings. Use DENSE_RANK when you want the Nth distinct value, like 'top 3 prices' where you want all items with those prices."


**对比示例：**

```sql
-- 数据: Alice=100, Bob=100, Carol=90, Dave=80
SELECT 
    name,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
    RANK()       OVER (ORDER BY score DESC) AS rank_val,
    DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank
FROM students;
```

| name | score | ROW_NUMBER | RANK | DENSE_RANK |
|------|-------|------------|------|------------|
| Alice | 100 | 1 | 1 | 1 |
| Bob | 100 | 2 | 1 | 1 |
| Carol | 90 | 3 | **3** | **2** |
| Dave | 80 | 4 | **4** | **3** |

**核心区别：**
- **ROW_NUMBER**：永远 1,2,3,4（相同值也不同序号）
- **RANK**：并列后跳过（1,1,3,4）
- **DENSE_RANK**：并列不跳（1,1,2,3）

**使用场景：**

```sql
-- 场景1：分页/去重 — 用 ROW_NUMBER（需要精确行数）
WITH numbered AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) AS rn
    FROM events
)
SELECT * FROM numbered WHERE rn = 1;  -- 每个用户最新一条

-- 场景2：比赛排名 — 用 RANK（并列第1后是第3名）
SELECT name, score, RANK() OVER (ORDER BY score DESC) AS position
FROM contestants;

-- 场景3：取前3种价格的所有商品 — 用 DENSE_RANK
WITH ranked AS (
    SELECT *, DENSE_RANK() OVER (ORDER BY price DESC) AS dr
    FROM products
)
SELECT * FROM ranked WHERE dr <= 3;  -- 可能返回超过3行
```


### Q4.2: How do you calculate a running total?

如何计算累计求和？


> "Use SUM as a window function with an ORDER BY clause and a frame specification. The default frame with ORDER BY is RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW, which sums from the first row to the current row. For explicit control, use ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW. The difference matters when you have ties — ROWS counts physical rows while RANGE groups equal values together. Always include ORDER BY; without it, you'll get the total sum for every row. You can partition the running total by groups using PARTITION BY, like calculating running totals per customer or per month."


**基本语法：**

```sql
SUM(column) OVER (
    [PARTITION BY 分组列]
    ORDER BY 排序列
    [ROWS/RANGE BETWEEN 起点 AND 终点]
)
```

**累计求和示例：**

```sql
-- 按日期计算累计销售额
SELECT 
    order_date,
    daily_sales,
    SUM(daily_sales) OVER (ORDER BY order_date) AS running_total
FROM daily_sales;

-- 结果:
-- | order_date | daily_sales | running_total |
-- | 2024-01-01 | 100         | 100           |
-- | 2024-01-02 | 150         | 250           |
-- | 2024-01-03 | 200         | 450           |
```

**按组累计（PARTITION BY）：**

```sql
-- 每个客户的累计消费
SELECT 
    customer_id,
    order_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY customer_id 
        ORDER BY order_date
    ) AS customer_running_total
FROM orders;
```

**ROWS vs RANGE 的区别：**

```sql
-- 当有相同日期时
-- | date       | value |
-- | 2024-01-01 | 10    |
-- | 2024-01-01 | 20    |  -- 相同日期
-- | 2024-01-02 | 30    |

-- ROWS: 按物理行计算（推荐）
SUM(value) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING)
-- 结果: 10, 30, 60

-- RANGE: 相同日期的值一起计算
SUM(value) OVER (ORDER BY date RANGE UNBOUNDED PRECEDING)
-- 结果: 30, 30, 60  -- 前两行结果相同
```


### Q4.3: How do you calculate a moving average?

如何计算移动平均？


> "Use AVG as a window function with a frame specification that defines the window size. For a 7-day moving average, use ROWS BETWEEN 6 PRECEDING AND CURRENT ROW — this includes the current row plus 6 previous rows. For a centered moving average, use ROWS BETWEEN N PRECEDING AND N FOLLOWING. Be careful with time-series data that has gaps — ROWS counts physical rows, not calendar days. If you have missing dates, first generate a complete date series and left join your data, filling gaps with zeros or nulls as appropriate. The frame size should be N-1 PRECEDING for an N-period average including the current row."


**移动平均语法：**

```sql
AVG(column) OVER (
    ORDER BY date_column
    ROWS BETWEEN N PRECEDING AND CURRENT ROW
)
```

**7天移动平均示例：**

```sql
SELECT 
    order_date,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW  -- 当前行 + 前6行 = 7天
    ) AS ma_7d
FROM daily_sales;
```

**居中移动平均（前后各取）：**

```sql
-- 3天居中移动平均：前1天 + 当天 + 后1天
SELECT 
    order_date,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY order_date
        ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
    ) AS centered_ma_3d
FROM daily_sales;
```

**处理日期间隙：**

```sql
-- 问题：如果日期有间隙，ROWS 会算错
-- 解决：先生成完整日期，再 LEFT JOIN

WITH date_series AS (
    SELECT generate_series(
        '2024-01-01'::date,
        '2024-01-31'::date,
        '1 day'::interval
    )::date AS dt
),
filled_data AS (
    SELECT 
        ds.dt,
        COALESCE(s.daily_sales, 0) AS daily_sales
    FROM date_series ds
    LEFT JOIN daily_sales s ON ds.dt = s.order_date
)
SELECT 
    dt,
    daily_sales,
    AVG(daily_sales) OVER (
        ORDER BY dt
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS ma_7d
FROM filled_data;
```




### Q4.4: Explain LAG and LEAD functions

解释 LAG 和 LEAD 函数


> "LAG and LEAD access data from other rows without a self-join. LAG looks at previous rows — LAG(column, 1) gets the immediately preceding value, LAG(column, 2) gets two rows back. LEAD looks at following rows in the same way. Both accept an optional default value for when the offset goes beyond the partition boundary. Common use cases include calculating row-over-row changes, period-over-period comparisons, and detecting gaps in sequences. They're much more efficient than self-joins for these operations. Remember they require an ORDER BY in the OVER clause to define which row is 'previous' or 'next'."


**基本语法：**

```sql
LAG(column, offset, default)  OVER (ORDER BY ...)  -- 取前面的行
LEAD(column, offset, default) OVER (ORDER BY ...)  -- 取后面的行
```

**参数说明：**
- `column`: 要获取的列
- `offset`: 偏移行数（默认 1）
- `default`: 超出范围时的默认值（默认 NULL）

**常用场景：**

```sql
-- 场景1：计算日环比变化
SELECT 
    order_date,
    daily_sales,
    LAG(daily_sales, 1) OVER (ORDER BY order_date) AS prev_day_sales,
    daily_sales - LAG(daily_sales, 1) OVER (ORDER BY order_date) AS day_over_day_change
FROM daily_sales;

-- 场景2：计算同比（去年同期）
SELECT 
    month,
    revenue,
    LAG(revenue, 12) OVER (ORDER BY month) AS last_year_revenue,
    (revenue - LAG(revenue, 12) OVER (ORDER BY month)) / 
        LAG(revenue, 12) OVER (ORDER BY month) * 100 AS yoy_growth_pct
FROM monthly_revenue;

-- 场景3：找出值发生变化的行
SELECT * FROM (
    SELECT *,
        LAG(status) OVER (PARTITION BY user_id ORDER BY event_time) AS prev_status
    FROM user_events
) t
WHERE status <> prev_status OR prev_status IS NULL;

-- 场景4：计算相邻行的时间间隔
SELECT 
    event_time,
    event_time - LAG(event_time) OVER (ORDER BY event_time) AS time_since_last_event
FROM events;
```

**提供默认值避免 NULL：**

```sql
SELECT 
    order_date,
    daily_sales,
    LAG(daily_sales, 1, 0) OVER (ORDER BY order_date) AS prev_sales  -- 第一行返回 0 而非 NULL
FROM daily_sales;
```

---



## 5. Subqueries and CTEs 

### Q5.1: What's the difference between correlated and non-correlated subqueries?

相关子查询和非相关子查询有什么区别？


> "A non-correlated subquery is independent — it can run on its own and returns the same result regardless of the outer query. It executes once and its result is used by the outer query. A correlated subquery references columns from the outer query, so it must re-execute for each row of the outer query — this can be expensive for large datasets. For example, 'WHERE salary > (SELECT AVG(salary) FROM employees)' is non-correlated, while 'WHERE salary > (SELECT AVG(salary) FROM employees e2 WHERE e2.department = e1.department)' is correlated because it references e1 from the outer query. Correlated subqueries can often be rewritten as JOINs for better performance."


**核心区别：**

| 特性 | 非相关子查询 | 相关子查询 |
|------|-------------|-----------|
| 独立性 | 可以独立运行 | 依赖外层查询 |
| 执行次数 | 执行 1 次 | 外层每行执行 1 次 |
| 性能 | 通常较好 | 可能较慢 |
| 识别方法 | 子查询不引用外层列 | 子查询引用外层列 |

**非相关子查询示例：**

```sql
-- 子查询独立执行一次，结果被外层使用
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees);

-- 执行过程：
-- 1. 执行子查询：SELECT AVG(salary) FROM employees → 5000
-- 2. 执行外层：SELECT ... WHERE salary > 5000
```

**相关子查询示例：**

```sql
-- 子查询引用了外层的 e1.department
SELECT e1.name, e1.salary, e1.department
FROM employees e1
WHERE e1.salary > (
    SELECT AVG(e2.salary) 
    FROM employees e2 
    WHERE e2.department = e1.department  -- 引用外层！
);

-- 执行过程（假设有 100 个员工）：
-- 对于每个员工，执行一次子查询计算该部门的平均工资
-- 总共执行 100 次子查询 → 性能差
```

**优化：将相关子查询改写为 JOIN**

```sql
-- 原始：相关子查询（慢）
SELECT e1.name, e1.salary
FROM employees e1
WHERE e1.salary > (
    SELECT AVG(e2.salary) FROM employees e2 WHERE e2.department = e1.department
);

-- 优化：JOIN（快）
SELECT e.name, e.salary
FROM employees e
INNER JOIN (
    SELECT department, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
) dept_avg ON e.department = dept_avg.department
WHERE e.salary > dept_avg.avg_salary;
```

**EXISTS 是常见的相关子查询：**

```sql
-- 找出有订单的客户
SELECT * FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o WHERE o.customer_id = c.id  -- 相关子查询
);

-- 这种情况 EXISTS 通常很高效，因为找到一条就停止
```


### Q5.2: What are the benefits of CTEs?

CTE 有什么好处？

> "CTEs — Common Table Expressions — defined with WITH clause offer several benefits. First, readability: they let you break complex queries into named, logical steps that read top-to-bottom. Second, reusability: you can reference the same CTE multiple times in a query without repeating the logic. Third, recursion: only CTEs support recursive queries for hierarchical data like org charts or bill of materials. Fourth, maintainability: when logic changes, you update it in one place. Compared to subqueries, CTEs are easier to debug — you can run each CTE independently. Compared to temp tables, CTEs don't require separate statements or cleanup, though temp tables are better when you need indexes or cross-statement reuse."


**CTE 语法：**

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT * FROM cte_name;
```

**好处 1：可读性（扁平化复杂查询）**

```sql
-- ❌ 嵌套子查询：难读
SELECT * FROM (
    SELECT * FROM (
        SELECT * FROM orders WHERE status = 'completed'
    ) t1 WHERE amount > 100
) t2 WHERE date > '2024-01-01';

-- ✅ CTE：清晰的逻辑步骤
WITH completed_orders AS (
    SELECT * FROM orders WHERE status = 'completed'
),
high_value AS (
    SELECT * FROM completed_orders WHERE amount > 100
)
SELECT * FROM high_value WHERE date > '2024-01-01';
```

**好处 2：可复用（同一查询中多次引用）**

```sql
-- 不用重复写相同的子查询
WITH monthly_sales AS (
    SELECT DATE_TRUNC('month', order_date) AS month, SUM(amount) AS total
    FROM orders
    GROUP BY DATE_TRUNC('month', order_date)
)
SELECT 
    a.month,
    a.total,
    b.total AS prev_month,
    a.total - b.total AS month_over_month
FROM monthly_sales a
LEFT JOIN monthly_sales b ON a.month = b.month + INTERVAL '1 month';
```

**好处 3：支持递归**

```sql
-- 递归查询：组织架构的所有下级
WITH RECURSIVE subordinates AS (
    -- 基础：直接下属
    SELECT id, name, manager_id, 1 AS level
    FROM employees
    WHERE manager_id = 1  -- 从 CEO 开始
    
    UNION ALL
    
    -- 递归：下属的下属
    SELECT e.id, e.name, e.manager_id, s.level + 1
    FROM employees e
    INNER JOIN subordinates s ON e.manager_id = s.id
)
SELECT * FROM subordinates;
```

**CTE vs 子查询 vs 临时表：**

| 特性 | CTE | 子查询 | 临时表 |
|------|-----|--------|--------|
| 可读性 | ⭐⭐⭐ | ⭐ | ⭐⭐ |
| 复用 | 同一语句内 | 不可 | 跨语句 |
| 递归 | ✅ | ❌ | ❌ |
| 索引 | ❌ | ❌ | ✅ |
| 物化 | 看数据库 | ❌ | ✅ |



### Q5.3: How do recursive CTEs work?

递归 CTE 是如何工作的？


> "A recursive CTE has two parts connected by UNION ALL. The anchor member runs first and provides the starting rows. The recursive member references the CTE itself and runs repeatedly, adding new rows each iteration until no more rows are produced. Each iteration only sees the rows from the previous iteration, not all accumulated rows. Common use cases include traversing hierarchies like employee-manager chains, generating sequences like date series, and expanding graph structures. Most databases have a recursion limit — PostgreSQL defaults to 100 — to prevent infinite loops. Always ensure your recursive condition will eventually produce no rows."


**递归 CTE 结构：**

```sql
WITH RECURSIVE cte_name AS (
    -- 锚点成员（Anchor）：起始数据
    SELECT ... 
    
    UNION ALL
    
    -- 递归成员（Recursive）：引用自己
    SELECT ... FROM cte_name WHERE 终止条件
)
SELECT * FROM cte_name;
```

**执行过程：**

```
第1次迭代: 执行锚点查询 → 结果集 R1
第2次迭代: 用 R1 执行递归查询 → 结果集 R2
第3次迭代: 用 R2 执行递归查询 → 结果集 R3
...
直到某次递归返回空结果集 → 停止

最终结果 = R1 ∪ R2 ∪ R3 ∪ ...
```

**示例 1：组织架构（找所有下属）**

```sql
WITH RECURSIVE org_tree AS (
    -- 锚点：起始员工
    SELECT id, name, manager_id, 1 AS level
    FROM employees
    WHERE id = 1  -- 从 CEO 开始
    
    UNION ALL
    
    -- 递归：找下属
    SELECT e.id, e.name, e.manager_id, t.level + 1
    FROM employees e
    INNER JOIN org_tree t ON e.manager_id = t.id
)
SELECT * FROM org_tree;

-- 执行过程：
-- 第1次: id=1 (CEO)
-- 第2次: CEO 的直接下属
-- 第3次: 那些下属的下属
-- ... 直到没有更多下属
```

**示例 2：生成日期序列**

```sql
WITH RECURSIVE date_series AS (
    -- 锚点：起始日期
    SELECT DATE '2024-01-01' AS dt
    
    UNION ALL
    
    -- 递归：加一天
    SELECT dt + INTERVAL '1 day'
    FROM date_series
    WHERE dt < DATE '2024-01-31'  -- 终止条件
)
SELECT * FROM date_series;
```

**示例 3：路径查找（图结构）**

```sql
-- 找从 A 到所有可达节点的路径
WITH RECURSIVE paths AS (
    SELECT 
        start_node,
        end_node,
        ARRAY[start_node, end_node] AS path,
        1 AS hops
    FROM edges
    WHERE start_node = 'A'
    
    UNION ALL
    
    SELECT 
        p.start_node,
        e.end_node,
        p.path || e.end_node,
        p.hops + 1
    FROM paths p
    INNER JOIN edges e ON p.end_node = e.start_node
    WHERE NOT e.end_node = ANY(p.path)  -- 避免循环
      AND p.hops < 10  -- 限制深度
)
SELECT * FROM paths;
```

**防止无限循环：**
- 设置深度限制（`WHERE level < 10`）
- 检测已访问节点（`WHERE node NOT IN (visited)`）
- 数据库有默认限制（PostgreSQL 100 层）

---



## 6. Performance Optimization 


### Q6.1: How do you analyze an EXPLAIN plan?

如何分析 EXPLAIN plan？


> "EXPLAIN shows how the database will execute your query. Key things to look for: Seq Scan indicates a full table scan — consider adding an index. Index Scan or Index Only Scan is usually good. Nested Loop with large tables can be slow — might need a Hash Join or Merge Join instead. Look at the cost estimates — higher numbers mean more work. With EXPLAIN ANALYZE, compare estimated versus actual row counts — big mismatches indicate stale statistics. Watch for Sort operations using disk instead of memory. The execution order is bottom-up and right-to-left in the plan. Focus on the highest-cost nodes first when optimizing."


**基本用法：**

```sql
-- 只看计划（不执行）
EXPLAIN SELECT * FROM orders WHERE customer_id = 123;

-- 实际执行并显示真实统计（PostgreSQL）
EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 123;

-- MySQL
EXPLAIN FORMAT=JSON SELECT * FROM orders WHERE customer_id = 123;
```

**关键指标：**

| 指标 | 含义 | 关注点 |
|------|------|--------|
| Seq Scan | 全表扫描 | 🔴 考虑加索引 |
| Index Scan | 索引扫描 | 🟢 正常 |
| Index Only Scan | 覆盖索引扫描 | 🟢 最优 |
| Nested Loop | 嵌套循环 JOIN | 小表 OK，大表 🔴 |
| Hash Join | 哈希 JOIN | 🟢 大表常用 |
| Sort | 排序操作 | 看是否用磁盘 |
| cost | 估算成本 | 越小越好 |
| rows | 估算行数 | 与实际比较 |

**EXPLAIN ANALYZE 输出示例：**

```sql
EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 123;

-- 输出：
Seq Scan on orders  (cost=0.00..1520.00 rows=50 width=100) 
                    (actual time=0.015..15.234 rows=48 loops=1)
  Filter: (customer_id = 123)
  Rows Removed by Filter: 49952

-- 分析：
-- Seq Scan: 全表扫描 → 需要在 customer_id 上加索引
-- rows=50 vs actual rows=48: 估算接近，统计信息正常
-- Rows Removed by Filter: 49952: 过滤了大量行 → 索引会很有帮助
```

**优化后：**

```sql
CREATE INDEX idx_orders_customer ON orders(customer_id);
EXPLAIN ANALYZE SELECT * FROM orders WHERE customer_id = 123;

-- 输出：
Index Scan using idx_orders_customer on orders  (cost=0.29..8.50 rows=50 width=100)
                                                (actual time=0.020..0.150 rows=48 loops=1)
  Index Cond: (customer_id = 123)

-- cost 从 1520 降到 8.5，time 从 15ms 降到 0.15ms
```

**常见问题及解决：**

| 问题 | 表现 | 解决方案 |
|------|------|----------|
| 全表扫描 | Seq Scan | 添加索引 |
| 估算不准 | estimated ≠ actual | `ANALYZE table_name` 更新统计 |
| 磁盘排序 | Sort Method: external | 增加 work_mem |
| 大表 Nested Loop | 耗时长 | 考虑 Hash Join |


### Q6.2: When do indexes become ineffective?

索引什么时候会失效？

> "Indexes become ineffective in several situations. First, using functions on indexed columns — WHERE YEAR(date_column) equals something prevents index use; rewrite as a range condition. Second, implicit type conversion — comparing a string column to a number forces conversion. Third, leading wildcards in LIKE — '%abc' can't use an index but 'abc%' can. Fourth, OR conditions may prevent index use — consider rewriting with UNION. Fifth, negations like NOT IN or NOT EQUALS often can't use indexes efficiently. Sixth, low selectivity — if a column has few distinct values like gender, a full scan might be faster. Seventh, very small tables — the optimizer may choose a scan over index overhead."


**索引失效的常见原因：**

**1. 在索引列上使用函数**

```sql
-- ❌ 索引失效
SELECT * FROM orders WHERE YEAR(order_date) = 2024;
SELECT * FROM users WHERE UPPER(email) = 'TEST@TEST.COM';

-- ✅ 改写为范围查询
SELECT * FROM orders 
WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01';

-- ✅ 或创建函数索引（PostgreSQL）
CREATE INDEX idx_users_email_upper ON users(UPPER(email));
```

**2. 隐式类型转换**

```sql
-- phone 是 VARCHAR 类型
-- ❌ 隐式转换导致索引失效
SELECT * FROM users WHERE phone = 1234567890;

-- ✅ 使用正确的类型
SELECT * FROM users WHERE phone = '1234567890';
```

**3. LIKE 以通配符开头**

```sql
-- ❌ 前导通配符无法用索引
SELECT * FROM users WHERE name LIKE '%john%';

-- ✅ 前缀匹配可以用索引
SELECT * FROM users WHERE name LIKE 'john%';
```

**4. OR 条件**

```sql
-- ❌ OR 可能无法有效使用索引
SELECT * FROM users WHERE status = 'A' OR status = 'B';

-- ✅ 使用 IN
SELECT * FROM users WHERE status IN ('A', 'B');

-- ✅ 或使用 UNION（某些情况）
SELECT * FROM users WHERE status = 'A'
UNION ALL
SELECT * FROM users WHERE status = 'B';
```

**5. 否定条件**

```sql
-- ❌ NOT IN / <> / != 通常无法用索引
SELECT * FROM users WHERE status <> 'deleted';
SELECT * FROM users WHERE id NOT IN (1, 2, 3);

-- ✅ 考虑反向逻辑
SELECT * FROM users WHERE status IN ('active', 'pending');
```

**6. 复合索引顺序错误**

```sql
-- 索引: (a, b, c)
-- ✅ 能用索引
WHERE a = 1
WHERE a = 1 AND b = 2
WHERE a = 1 AND b = 2 AND c = 3

-- ❌ 不能有效用索引（跳过了 a）
WHERE b = 2
WHERE b = 2 AND c = 3

-- 部分使用索引
WHERE a = 1 AND c = 3  -- 只用到 a 部分
```

**7. 数据分布问题**

```sql
-- 如果 gender 只有 'M'/'F' 两个值
-- 选择性太低，索引可能不如全表扫描
SELECT * FROM users WHERE gender = 'M';  -- 返回 50% 数据

-- 一般规则：索引选择性 > 10-15% 才有意义
```

**索引失效检查清单：**

| 检查项 | 问题 | 解决 |
|--------|------|------|
| 列上有函数？ | `YEAR(date)` | 改为范围查询 |
| 类型匹配？ | 字符串 vs 数字 | 使用正确类型 |
| LIKE 前导 %？ | `'%abc'` | 考虑全文搜索 |
| 复合索引顺序？ | 跳过前导列 | 遵循最左前缀 |
| 选择性够高？ | < 10% 区分度 | 可能不需要索引 |


### Q6.3: How do you optimize a slow query?

如何优化慢查询？


> "I follow a systematic approach. First, use EXPLAIN ANALYZE to understand what's happening — look for full scans, high costs, and row estimate mismatches. Second, ensure proper indexes exist for WHERE, JOIN, and ORDER BY columns. Third, check for query anti-patterns: SELECT star, functions on indexed columns, unnecessary DISTINCT. Fourth, restructure the query — replace correlated subqueries with JOINs, use EXISTS instead of IN for large datasets. Fifth, filter early — push WHERE conditions as close to the base tables as possible. Sixth, for analytical queries, consider partitioning, materialized views, or pre-aggregation. The biggest wins usually come from proper indexing and eliminating unnecessary work."


**优化步骤：**

**Step 1: 诊断 — EXPLAIN ANALYZE**

```sql
EXPLAIN ANALYZE SELECT ...;

-- 关注：
-- - Seq Scan（全表扫描）
-- - 高 cost 节点
-- - estimated vs actual rows 差异大
-- - Sort 使用磁盘
```

**Step 2: 检查索引**

```sql
-- 确保 WHERE、JOIN、ORDER BY 列有索引
CREATE INDEX idx_orders_customer ON orders(customer_id);
CREATE INDEX idx_orders_date ON orders(order_date);

-- 复合索引用于多条件查询
CREATE INDEX idx_orders_customer_date ON orders(customer_id, order_date);
```

**Step 3: 消除反模式**

```sql
-- ❌ SELECT *
SELECT * FROM orders WHERE customer_id = 123;
-- ✅ 只选需要的列
SELECT order_id, order_date, total FROM orders WHERE customer_id = 123;

-- ❌ 在索引列上用函数
WHERE YEAR(order_date) = 2024
-- ✅ 范围查询
WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01'

-- ❌ 不必要的 DISTINCT
SELECT DISTINCT customer_id FROM orders WHERE status = 'completed';
-- ✅ 检查是否真的需要去重
```

**Step 4: 重写查询**

```sql
-- ❌ 相关子查询（每行执行一次）
SELECT * FROM orders o
WHERE total > (SELECT AVG(total) FROM orders WHERE customer_id = o.customer_id);

-- ✅ 改为 JOIN（执行一次）
SELECT o.* 
FROM orders o
JOIN (
    SELECT customer_id, AVG(total) AS avg_total
    FROM orders
    GROUP BY customer_id
) avg ON o.customer_id = avg.customer_id
WHERE o.total > avg.avg_total;

-- ❌ IN + 大子查询
WHERE customer_id IN (SELECT id FROM customers WHERE country = 'US');
-- ✅ EXISTS（找到即停）
WHERE EXISTS (SELECT 1 FROM customers c WHERE c.id = customer_id AND c.country = 'US');
```

**Step 5: 架构级优化**

```sql
-- 分区（大表）
CREATE TABLE orders (...) PARTITION BY RANGE (order_date);

-- 物化视图（复杂聚合）
CREATE MATERIALIZED VIEW daily_stats AS
SELECT date, SUM(amount), COUNT(*) FROM orders GROUP BY date;

-- 更新统计信息
ANALYZE orders;
```

**优化检查清单：**

| 优化项 | 检查 | 操作 |
|--------|------|------|
| 索引 | 是否存在？是否被使用？ | EXPLAIN + CREATE INDEX |
| SELECT | 是否用了 *？ | 只选需要的列 |
| WHERE | 列上有函数？类型正确？ | 重写条件 |
| JOIN | 是否有相关子查询？ | 改为 JOIN |
| 统计 | 估算是否准确？ | ANALYZE 更新 |

---



# 🔴 SENIOR SQL QUESTIONS 

## 7. Advanced Optimization 



### Q7.1: Explain covering indexes

解释覆盖索引的原理


> "A covering index includes all columns needed by a query, so the database can satisfy the query entirely from the index without accessing the main table — this is called an 'index-only scan.' It's particularly valuable for frequently-run queries on wide tables. In PostgreSQL, use INCLUDE to add non-key columns to an index. In SQL Server and MySQL, you can include columns in the index definition. The trade-off is index size — covering indexes are larger and slower to update. Use them for read-heavy queries where the same columns are consistently accessed. Monitor index usage and size to ensure the overhead is justified."


**什么是覆盖索引？**
- 索引包含查询需要的**所有列**
- 查询可以完全从索引获取数据，**无需回表**
- EXPLAIN 中显示为 `Index Only Scan`

**回表 vs 覆盖索引：**

```text
普通索引查询：
Index Scan → 找到符合条件的行 ID → 回到主表读取完整行 → 返回结果
                                    ↑
                               这一步叫"回表"

覆盖索引查询：
Index Only Scan → 直接从索引获取所有需要的列 → 返回结果
                  ↑
              不需要回表
```

**创建覆盖索引：**

```sql
-- PostgreSQL: 使用 INCLUDE 添加非键列
CREATE INDEX idx_orders_covering ON orders(customer_id) 
INCLUDE (order_date, total_amount);

-- 现在这个查询可以使用 Index Only Scan
SELECT order_date, total_amount FROM orders WHERE customer_id = 123;

-- MySQL: 直接把列加入索引
CREATE INDEX idx_orders_covering ON orders(customer_id, order_date, total_amount);

-- SQL Server: 使用 INCLUDE
CREATE INDEX idx_orders_covering ON orders(customer_id) 
INCLUDE (order_date, total_amount);
```

**何时使用覆盖索引：**

| 场景 | 适合 |
|------|------|
| 读多写少的查询 | ✅ |
| 频繁执行的查询 | ✅ |
| 宽表只取少数列 | ✅ |
| 写入频繁的表 | ❌ 索引维护开销大 |
| 需要列经常变化 | ❌ |

**注意事项：**
- 覆盖索引更大，占用更多存储
- 写入时需要维护更多索引数据
- 需要权衡读取收益 vs 写入成本



### Q7.2: What are query rewriting techniques?

有哪些查询重写技巧？


> "Query rewriting transforms logically equivalent queries into more efficient forms. Key techniques include: replacing correlated subqueries with JOINs for single execution; using EXISTS instead of IN for large subqueries since EXISTS stops at first match; converting OR conditions to UNION for better index usage; pushing predicates into subqueries or CTEs to filter early; using semi-joins with EXISTS instead of DISTINCT after JOIN; replacing NOT IN with NOT EXISTS to handle NULLs correctly and often perform better; and using derived tables to pre-aggregate before joining. The goal is reducing the amount of data processed at each step while maintaining the same results."


**技巧 1：相关子查询 → JOIN**

```sql
-- ❌ 慢：每行执行一次子查询
SELECT * FROM orders o
WHERE o.amount > (
    SELECT AVG(amount) FROM orders WHERE customer_id = o.customer_id
);

-- ✅ 快：只执行一次
SELECT o.* FROM orders o
JOIN (
    SELECT customer_id, AVG(amount) AS avg_amt
    FROM orders GROUP BY customer_id
) t ON o.customer_id = t.customer_id
WHERE o.amount > t.avg_amt;
```

**技巧 2：IN → EXISTS（大数据集）**

```sql
-- ❌ IN: 需要完整执行子查询
SELECT * FROM orders
WHERE customer_id IN (SELECT id FROM customers WHERE country = 'US');

-- ✅ EXISTS: 找到一条就停止
SELECT * FROM orders o
WHERE EXISTS (
    SELECT 1 FROM customers c WHERE c.id = o.customer_id AND c.country = 'US'
);
```

**技巧 3：NOT IN → NOT EXISTS（避免 NULL 问题）**

```sql
-- ❌ NOT IN 遇到 NULL 会返回空结果！
SELECT * FROM orders
WHERE customer_id NOT IN (SELECT id FROM deleted_customers);  -- 如果有 NULL，返回空

-- ✅ NOT EXISTS 正确处理 NULL
SELECT * FROM orders o
WHERE NOT EXISTS (
    SELECT 1 FROM deleted_customers d WHERE d.id = o.customer_id
);
```

**技巧 4：OR → UNION**

```sql
-- ❌ OR 可能无法有效使用索引
SELECT * FROM products WHERE category = 'A' OR brand = 'Nike';

-- ✅ UNION 可以分别使用各自的索引
SELECT * FROM products WHERE category = 'A'
UNION
SELECT * FROM products WHERE brand = 'Nike';
```

**技巧 5：DISTINCT + JOIN → EXISTS（去重）**

```sql
-- ❌ 先 JOIN 产生重复，再 DISTINCT 去重
SELECT DISTINCT c.* FROM customers c
JOIN orders o ON c.id = o.customer_id;

-- ✅ 用 EXISTS 避免产生重复
SELECT * FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.id);
```

**技巧 6：提前过滤（Predicate Pushdown）**

```sql
-- ❌ 先 JOIN 全部数据，再过滤
SELECT * FROM orders o
JOIN customers c ON o.customer_id = c.id
WHERE o.order_date > '2024-01-01';

-- ✅ 先过滤，再 JOIN（优化器通常会自动做）
SELECT * FROM (SELECT * FROM orders WHERE order_date > '2024-01-01') o
JOIN customers c ON o.customer_id = c.id;
```



### Q7.3: How do you design table partitioning?

如何设计分区表？

> "Partitioning divides a large table into smaller, more manageable pieces while appearing as a single table. Choose the partition key based on your most common query patterns — typically date for time-series data. Range partitioning is most common for dates; list partitioning for discrete values like regions; hash partitioning for even distribution. Benefits include faster queries through partition pruning, easier maintenance like dropping old data, and parallel processing. Design considerations: choose a key that's in most WHERE clauses, size partitions appropriately — not too many small ones, ensure queries include the partition key to enable pruning. Don't over-partition; monitor query plans to confirm pruning is happening."


**什么是分区？**
- 将大表物理拆分为多个小表
- 对应用透明，查询时仍是一张表
- 根据分区键自动路由到正确的分区

**三种分区类型：**

| 类型 | 适用场景 | 示例 |
|------|----------|------|
| **Range** | 时间序列、连续值 | 按月/年分区 |
| **List** | 离散值、枚举 | 按地区、状态分区 |
| **Hash** | 均匀分布、无明显模式 | 按 user_id 哈希 |

**Range 分区示例（最常用）：**

```sql
-- PostgreSQL
CREATE TABLE orders (
    id SERIAL,
    order_date DATE,
    amount DECIMAL(10,2)
) PARTITION BY RANGE (order_date);

-- 创建分区
CREATE TABLE orders_2024_q1 PARTITION OF orders
    FOR VALUES FROM ('2024-01-01') TO ('2024-04-01');
CREATE TABLE orders_2024_q2 PARTITION OF orders
    FOR VALUES FROM ('2024-04-01') TO ('2024-07-01');

-- 查询自动路由到正确分区
SELECT * FROM orders WHERE order_date = '2024-02-15';
-- 只扫描 orders_2024_q1 分区
```

**List 分区示例：**

```sql
CREATE TABLE sales (
    id SERIAL,
    region VARCHAR(20),
    amount DECIMAL(10,2)
) PARTITION BY LIST (region);

CREATE TABLE sales_asia PARTITION OF sales FOR VALUES IN ('CN', 'JP', 'KR');
CREATE TABLE sales_europe PARTITION OF sales FOR VALUES IN ('UK', 'DE', 'FR');
CREATE TABLE sales_america PARTITION OF sales FOR VALUES IN ('US', 'CA', 'MX');
```

**分区的好处：**

| 好处 | 说明 |
|------|------|
| 查询加速 | 分区裁剪（Partition Pruning）只扫描相关分区 |
| 维护方便 | 删除旧数据 = 删除分区（秒级 vs 逐行删除） |
| 并行处理 | 可以并行扫描多个分区 |
| 存储管理 | 不同分区可放不同存储介质 |

**设计原则：**
- 分区键选择最常用的查询条件
- 确保查询包含分区键（否则扫描所有分区）
- 分区数量适中（太多会增加规划开销）
- 定期创建新分区、归档旧分区



### Q7.4: When do you use materialized views?
什么时候使用物化视图？

> "Materialized views store precomputed query results physically, trading storage for query speed. Use them when: you have expensive aggregations or joins that run frequently; the underlying data changes infrequently compared to read frequency; slight staleness is acceptable; and query performance is critical. They're ideal for dashboards, reports, and analytics queries. The key decision is refresh strategy — on-commit for near real-time at write cost, periodic refresh for batch scenarios, or incremental refresh if supported. Don't use them for highly volatile data or when freshness is critical. Monitor refresh times and storage costs to ensure the trade-off remains beneficial."


**什么是物化视图？**
- 普通视图：每次查询时执行定义的 SQL
- 物化视图：**预先计算并存储**结果，查询直接读取

**适用场景：**

| 场景 | 原因 |
|------|------|
| 复杂聚合报表 | 避免每次重新计算 |
| Dashboard 查询 | 快速响应，允许轻微延迟 |
| 数据仓库宽表 | 预先 JOIN 多个表 |
| 跨大表的统计 | 减少扫描时间 |

**创建和使用：**

```sql
-- 创建物化视图
CREATE MATERIALIZED VIEW daily_sales_mv AS
SELECT 
    DATE(order_date) AS date,
    product_id,
    SUM(quantity) AS total_quantity,
    SUM(amount) AS total_revenue
FROM orders
JOIN order_items ON orders.id = order_items.order_id
WHERE status = 'completed'
GROUP BY DATE(order_date), product_id;

-- 查询物化视图（非常快）
SELECT * FROM daily_sales_mv WHERE date = '2024-01-15';

-- 刷新物化视图
REFRESH MATERIALIZED VIEW daily_sales_mv;

-- 并发刷新（不阻塞读取，需要 UNIQUE 索引）
REFRESH MATERIALIZED VIEW CONCURRENTLY daily_sales_mv;
```

**刷新策略：**

| 策略 | 说明 | 适用 |
|------|------|------|
| 手动刷新 | 需要时手动执行 | 简单场景 |
| 定时刷新 | cron job 定期刷新 | 报表（每小时/每天） |
| 触发器刷新 | 数据变更时刷新 | 近实时（写入开销大） |
| 增量刷新 | 只更新变化部分（部分数据库支持） | 大数据量 |

**物化视图 vs 普通表 vs 视图：**

| 特性 | 普通视图 | 物化视图 | 普通表 |
|------|----------|----------|--------|
| 存储数据 | ❌ | ✅ | ✅ |
| 查询速度 | 取决于原查询 | 快 | 快 |
| 数据新鲜度 | 实时 | 刷新时更新 | 手动维护 |
| 维护成本 | 无 | 自动/定时刷新 | 需要 ETL |

---



## 8. Transactions and Concurrency 




### Q8.1: Explain the four isolation levels

解释四种隔离级别


> "Isolation levels control how transactions see each other's changes. Read Uncommitted allows dirty reads — you can see uncommitted changes that might roll back. Read Committed, the most common default, only sees committed data but the same query might return different results within a transaction. Repeatable Read guarantees consistent reads within a transaction but phantom rows can appear — new rows matching your query criteria inserted by other transactions. Serializable provides complete isolation — transactions execute as if serial, preventing all anomalies but with lowest concurrency. Higher isolation means more consistency but less concurrency and potential for deadlocks."


**四种隔离级别及其防止的问题：**

| 隔离级别 | 脏读 | 不可重复读 | 幻读 |
|----------|------|------------|------|
| READ UNCOMMITTED | ❌ 可能 | ❌ 可能 | ❌ 可能 |
| READ COMMITTED | ✅ 防止 | ❌ 可能 | ❌ 可能 |
| REPEATABLE READ | ✅ 防止 | ✅ 防止 | ❌ 可能 |
| SERIALIZABLE | ✅ 防止 | ✅ 防止 | ✅ 防止 |

**三种并发问题：**


- 脏读（Dirty Read）：读到未提交的数据
  - 事务A: UPDATE balance = 0 (未提交)
  - 事务B: SELECT balance → 看到 0
  - 事务A: ROLLBACK → 0 从未存在，事务B 读到了"脏"数据
- 不可重复读（Non-repeatable Read）：同一事务两次读取不同
  - 事务A: SELECT balance → 100
  - 事务B: UPDATE balance = 50; COMMIT
  - 事务A: SELECT balance → 50（同一事务内不一致！）
- 幻读（Phantom Read）：同一查询返回不同行数
  - 事务A: SELECT COUNT(*) WHERE age > 20 → 10
  - 事务B: INSERT INTO ... (age = 25); COMMIT
  - 事务A: SELECT COUNT(*) WHERE age > 20 → 11（幻影行！）


**设置隔离级别：**

```sql
-- PostgreSQL
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;
BEGIN;
-- queries
COMMIT;

-- MySQL
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;
START TRANSACTION;
-- queries
COMMIT;
```

**各数据库默认隔离级别：**

| 数据库 | 默认级别 |
|--------|----------|
| PostgreSQL | READ COMMITTED |
| MySQL (InnoDB) | REPEATABLE READ |
| SQL Server | READ COMMITTED |
| Oracle | READ COMMITTED |


### Q8.2: How do deadlocks occur and how do you resolve them?

死锁如何产生？如何解决？


> "A deadlock occurs when two or more transactions are waiting for each other to release locks, creating a circular dependency. For example, transaction A locks row 1 and waits for row 2, while transaction B locks row 2 and waits for row 1 — neither can proceed. Databases detect this and automatically roll back one transaction, the 'victim.' To prevent deadlocks: access resources in consistent order across all transactions; keep transactions short; use appropriate isolation levels; avoid user interaction during transactions; and use NOWAIT or timeouts to fail fast. When deadlocks occur, retry the rolled-back transaction. Monitor deadlock frequency — high rates indicate design issues."


**死锁是如何产生的？**

```text
事务 A                        事务 B
────────                      ────────
锁定 row 1 ✓
                              锁定 row 2 ✓
请求锁定 row 2 （等待B）
                              请求锁定 row 1 （等待A）
        
        ↓ 互相等待，谁也无法继续 = 死锁
```

**死锁示例：**

```sql
-- 事务 A
BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;  -- 锁定账户1
-- 等待...
UPDATE accounts SET balance = balance + 100 WHERE id = 2;  -- 等待账户2

-- 事务 B（同时执行）
BEGIN;
UPDATE accounts SET balance = balance - 50 WHERE id = 2;   -- 锁定账户2
-- 等待...
UPDATE accounts SET balance = balance + 50 WHERE id = 1;   -- 等待账户1

-- 死锁！数据库会选择一个事务回滚
```

**预防死锁：**

| 方法 | 说明 |
|------|------|
| **固定顺序访问** | 所有事务按相同顺序锁定资源（如按 ID 升序） |
| **缩短事务** | 减少持锁时间 |
| **使用 NOWAIT** | 拿不到锁立即失败，而非等待 |
| **批量处理** | 一次锁定所有需要的行 |
| **降低隔离级别** | 减少锁的使用 |

**解决死锁：**

```sql
-- 预防：固定顺序（按 ID 排序）
BEGIN;
-- 总是先锁 ID 小的
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;
COMMIT;

-- 使用 NOWAIT 避免长时间等待
SELECT * FROM accounts WHERE id = 1 FOR UPDATE NOWAIT;  -- 拿不到锁立即报错

-- 设置锁超时
SET lock_timeout = '5s';  -- PostgreSQL
```

**处理死锁异常：**

```python
# 应用层重试逻辑
for attempt in range(3):
    try:
        execute_transaction()
        break
    except DeadlockError:
        if attempt < 2:
            time.sleep(random.uniform(0.1, 0.5))  # 随机等待后重试
        else:
            raise
```




### Q8.3: Explain optimistic vs pessimistic locking

解释乐观锁和悲观锁

> "Pessimistic locking assumes conflicts are likely, so it locks resources upfront using SELECT FOR UPDATE. The lock is held until transaction commit, preventing others from modifying the data. It's simple but reduces concurrency. Optimistic locking assumes conflicts are rare, so it doesn't lock during the transaction. Instead, it checks at commit time whether the data changed — typically using a version number or timestamp. If changed, the transaction fails and must retry. Use pessimistic locking for high-contention scenarios like inventory systems. Use optimistic locking for low-contention scenarios with many reads, like user profile updates. Optimistic scales better but requires retry logic."


**核心区别：**

| 特性 | 悲观锁 | 乐观锁 |
|------|--------|--------|
| 假设 | 冲突很可能发生 | 冲突很少发生 |
| 加锁时机 | 读取时立即加锁 | 不加锁，提交时检查 |
| 实现方式 | `SELECT FOR UPDATE` | 版本号/时间戳 |
| 并发度 | 低 | 高 |
| 复杂度 | 简单，数据库处理 | 需要应用层重试逻辑 |

**悲观锁示例：**

```sql
-- 库存扣减（高并发场景）
BEGIN;
-- 锁定该行，其他事务无法修改
SELECT stock FROM products WHERE id = 1 FOR UPDATE;

-- 检查库存
-- 如果足够，扣减
UPDATE products SET stock = stock - 1 WHERE id = 1;
COMMIT;

-- 变体：NOWAIT（拿不到锁立即失败）
SELECT * FROM products WHERE id = 1 FOR UPDATE NOWAIT;

-- 变体：SKIP LOCKED（跳过已锁定的行，用于队列处理）
SELECT * FROM jobs WHERE status = 'pending' 
FOR UPDATE SKIP LOCKED LIMIT 1;
```

**乐观锁示例：**

```sql
-- 表结构：添加版本号列
CREATE TABLE products (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    stock INT,
    version INT DEFAULT 0  -- 版本号
);

-- 更新时检查版本号
UPDATE products 
SET stock = stock - 1, 
    version = version + 1
WHERE id = 1 AND version = 5;  -- 版本号必须匹配

-- 检查影响行数
-- 如果是 0，说明被其他事务修改过，需要重试
```

**应用层乐观锁处理：**

```python
def update_stock(product_id, quantity):
    for attempt in range(3):
        # 读取当前版本
        product = db.query("SELECT * FROM products WHERE id = ?", product_id)
        current_version = product.version
        
        # 尝试更新
        rows_affected = db.execute("""
            UPDATE products 
            SET stock = stock - ?, version = version + 1
            WHERE id = ? AND version = ?
        """, quantity, product_id, current_version)
        
        if rows_affected > 0:
            return True  # 成功
        # 版本冲突，重试
    
    raise Exception("Update failed after retries")
```

**选择建议：**

| 场景 | 推荐 |
|------|------|
| 库存扣减、票务系统 | 悲观锁（高并发写入） |
| 用户资料更新 | 乐观锁（低冲突） |
| 长事务 | 乐观锁（避免长时间锁定） |
| 简单实现 | 悲观锁 |

---



## 9. Architecture Design 

### Q9.1: How do you decide between normalization and denormalization?

如何在规范化和反规范化之间做选择？

> "Normalization eliminates redundancy and ensures data integrity through normal forms — it's ideal for OLTP systems with many writes. Denormalization intentionally adds redundancy to improve read performance — it's common in OLAP and reporting systems. The decision depends on your read-write ratio: write-heavy transactional systems favor normalization to avoid update anomalies; read-heavy analytical systems favor denormalization to avoid expensive joins. In practice, start normalized for data integrity, then selectively denormalize based on performance needs — use materialized views, summary tables, or caching rather than wholesale denormalization. Document denormalization decisions and implement triggers or application logic to maintain consistency."


**规范化 vs 反规范化：**

| 特性 | 规范化 | 反规范化 |
|------|--------|----------|
| 数据冗余 | 最小化 | 允许冗余 |
| 写入性能 | 好（更新一处） | 差（多处更新） |
| 读取性能 | 可能差（需要 JOIN） | 好（减少 JOIN） |
| 数据一致性 | 强 | 需要额外维护 |
| 存储空间 | 少 | 多 |
| 适用场景 | OLTP（事务处理） | OLAP（分析查询） |

**规范化示例（3NF）：**

```sql
-- 规范化设计：消除冗余
-- 订单表
CREATE TABLE orders (
    id INT PRIMARY KEY,
    customer_id INT REFERENCES customers(id),
    order_date DATE
);

-- 客户表（客户信息只存一处）
CREATE TABLE customers (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    email VARCHAR(100)
);

-- 查询需要 JOIN
SELECT o.id, c.name, c.email, o.order_date
FROM orders o
JOIN customers c ON o.customer_id = c.id;
```

**反规范化示例：**

```sql
-- 反规范化：把常用字段冗余到订单表
CREATE TABLE orders_denormalized (
    id INT PRIMARY KEY,
    customer_id INT,
    customer_name VARCHAR(100),  -- 冗余！
    customer_email VARCHAR(100), -- 冗余！
    order_date DATE
);

-- 查询不需要 JOIN
SELECT id, customer_name, customer_email, order_date
FROM orders_denormalized;

-- 但是：客户改名时需要更新所有订单！
UPDATE orders_denormalized 
SET customer_name = 'New Name' 
WHERE customer_id = 123;
```

**决策框架：**

| 问题 | 答案导向 |
|------|----------|
| 读写比例？ | 读多 → 反规范化；写多 → 规范化 |
| 数据一致性要求？ | 高 → 规范化 |
| 查询复杂度？ | 需要多表 JOIN → 考虑反规范化 |
| 数据变更频率？ | 高 → 规范化 |

**实践建议：**
1. 从规范化开始（保证正确性）
2. 识别性能瓶颈（通过监控）
3. 选择性反规范化（只针对热点查询）
4. 使用物化视图（而非直接冗余）
5. 记录反规范化决策和同步策略



### Q9.2: How do you choose a sharding strategy?

如何选择分片策略？

> "Sharding distributes data across multiple databases for horizontal scaling. Key strategies: Range sharding divides by value ranges like date or ID — simple but can create hotspots. Hash sharding distributes evenly using a hash function — good load balance but range queries span all shards. Directory-based sharding uses a lookup service — flexible but adds a dependency. Choose your shard key carefully: it should be in most queries, have high cardinality, and distribute writes evenly. Avoid sharding unless necessary — it adds complexity for joins, transactions, and resharding. Consider your query patterns: if most queries need data from multiple shards, sharding may hurt more than help."


**什么是分片（Sharding）？**
- 将数据水平拆分到多个数据库实例
- 每个分片只存储部分数据
- 应用层或中间件路由请求

**三种分片策略：**

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **Range** | 按范围分（如日期、ID） | 范围查询高效 | 可能热点不均 |
| **Hash** | 哈希函数分配 | 分布均匀 | 范围查询需查所有分片 |
| **Directory** | 查表确定分片 | 灵活 | 单点依赖 |

**Range 分片示例：**

```text
Shard 1: user_id 1-1000000
Shard 2: user_id 1000001-2000000
Shard 3: user_id 2000001-3000000

优点：范围查询 WHERE user_id BETWEEN 1 AND 500000 只查 Shard 1
缺点：新用户都写入最新分片，可能成为热点
```

**Hash 分片示例：**

```text
Shard = hash(user_id) % 3

user_id=1 → Shard 1
user_id=2 → Shard 2
user_id=3 → Shard 0
user_id=4 → Shard 1

优点：数据分布均匀
缺点：范围查询需要查所有分片
```

**选择分片键的原则：**

| 原则 | 原因 |
|------|------|
| 出现在大多数查询中 | 否则需要跨分片查询 |
| 高基数（多个不同值） | 低基数无法有效分布 |
| 写入分布均匀 | 避免热点 |
| 不频繁变更 | 变更需要迁移数据 |

**分片带来的挑战：**

```sql
-- 挑战1：跨分片 JOIN
-- 如果 orders 按 user_id 分片，products 按 product_id 分片
-- 这个 JOIN 需要跨所有分片
SELECT * FROM orders o JOIN products p ON o.product_id = p.id;

-- 挑战2：跨分片事务
-- ACID 事务跨多个数据库很复杂（需要两阶段提交）

-- 挑战3：聚合查询
-- 需要在应用层合并各分片结果
SELECT COUNT(*) FROM users;  -- 需要 SUM 各分片的 COUNT
```

**何时分片？**
- 单机无法承载数据量
- 单机无法承载写入量
- 需要地理分布（就近访问）

**替代方案（优先考虑）：**
- 读写分离
- 垂直拆分（按业务分库）
- 分区表
- 缓存层



### Q9.3: How do you implement read-write separation?
如何实现读写分离？


> "Read-write separation directs writes to a primary database and reads to one or more replicas, scaling read capacity. Implementation approaches: application-level routing where code explicitly chooses connections; middleware or proxy like ProxySQL that routes based on query type; or ORM configuration with separate read and write connections. Key challenges include replication lag — reads might see stale data — and handling read-after-write consistency where a user should see their own changes. Solutions include: sticky sessions to route user requests to primary briefly after writes, explicit primary reads for critical queries, or accepting eventual consistency. Monitor replication lag and have fallback logic."


**什么是读写分离？**

```text
         应用程序
            │
     ┌──────┴──────┐
     ↓ 写          ↓ 读
  [Primary]  →  [Replica 1]
     │              │
     └───复制────────┤
                 [Replica 2]
```

**实现方式：**

| 方式 | 说明 | 适用 |
|------|------|------|
| 应用层路由 | 代码中指定读/写连接 | 简单场景 |
| 中间件代理 | ProxySQL、MaxScale | 透明，复杂场景 |
| ORM 配置 | 框架支持的读写分离 | Django, Rails 等 |

**应用层实现示例：**

```python
class DatabaseRouter:
    def get_write_connection(self):
        return connect("primary.db.com")
    
    def get_read_connection(self):
        return connect(random.choice([
            "replica1.db.com",
            "replica2.db.com"
        ]))

# 使用
db = DatabaseRouter()
db.get_write_connection().execute("INSERT INTO ...")
db.get_read_connection().execute("SELECT ...")
```

**主要挑战：复制延迟**

```
时间线：
T1: 用户更新个人资料 (写 Primary)
T2: 用户刷新页面 (读 Replica) → 看到旧数据！
T3: 复制完成
T4: 用户再次刷新 → 看到新数据

这就是"读己之写"（Read-Your-Writes）问题
```

**解决复制延迟的方案：**

| 方案 | 说明 |
|------|------|
| **写后读主库** | 写操作后短时间内强制读主库 |
| **会话粘性** | 写后一段时间内所有请求路由到主库 |
| **版本标记** | 记录写入版本，读时检查复制进度 |
| **接受最终一致** | 业务允许短暂延迟 |

**写后读主库示例：**

```python
def update_profile(user_id, data):
    db.get_write_connection().execute("UPDATE users SET ... WHERE id = ?", user_id)
    # 标记该用户需要读主库
    cache.set(f"read_primary:{user_id}", True, ttl=5)  # 5秒后过期

def get_profile(user_id):
    if cache.get(f"read_primary:{user_id}"):
        return db.get_write_connection().execute("SELECT * FROM users WHERE id = ?", user_id)
    else:
        return db.get_read_connection().execute("SELECT * FROM users WHERE id = ?", user_id)
```

**监控要点：**
- 复制延迟（Seconds Behind Master）
- 各副本的查询分布
- 主库写入压力
- 副本健康状态



---

## Quick Reference 

| Level | Topic | Key Points |
|-------|-------|------------|
| 🟢 Junior | SELECT/WHERE/ORDER BY | 执行顺序：FROM→WHERE→SELECT→ORDER BY→LIMIT |
| 🟢 Junior | DISTINCT vs GROUP BY | DISTINCT 去重，GROUP BY 聚合 |
| 🟢 Junior | NULL handling | IS NULL/IS NOT NULL, COALESCE |
| 🟢 Junior | LIKE patterns | % 多字符，_ 单字符，前导%无法用索引 |
| 🟢 Junior | Date functions | 避免在列上用函数，用范围查询 |
| 🟢 Junior | JOIN types | INNER/LEFT/RIGHT/FULL，LEFT JOIN + IS NULL 找不匹配 |
| 🟢 Junior | Multi-table JOIN | 链式 JOIN，注意类型混用影响 |
| 🟢 Junior | Self JOIN | 同表不同别名，层级/配对查询 |
| 🟢 Junior | Aggregates | COUNT(*) vs COUNT(col)，AVG 忽略 NULL |
| 🟢 Junior | WHERE vs HAVING | WHERE 过滤行，HAVING 过滤组 |
| 🟢 Junior | Finding duplicates | GROUP BY HAVING，窗口函数 |
| 🟡 Mid | ROW_NUMBER/RANK/DENSE_RANK | 唯一序号/跳过/不跳过 |
| 🟡 Mid | Running total | SUM() OVER (ORDER BY ...) |
| 🟡 Mid | Moving average | AVG() OVER (ROWS BETWEEN n PRECEDING AND CURRENT ROW) |
| 🟡 Mid | LAG/LEAD | 访问前/后行，计算变化 |
| 🟡 Mid | Correlated subquery | 依赖外层，每行执行，考虑改 JOIN |
| 🟡 Mid | CTE benefits | 可读性、复用、递归支持 |
| 🟡 Mid | Recursive CTE | 锚点 + 递归 + UNION ALL |
| 🟡 Mid | EXPLAIN analysis | 看 Seq Scan、cost、rows 估算 |
| 🟡 Mid | Index ineffective | 函数、类型转换、前导%、OR |
| 🟡 Mid | Query optimization | 索引 → 重写 → 架构 |
| 🔴 Senior | Covering index | 包含所有查询列，Index Only Scan |
| 🔴 Senior | Query rewriting | 相关子查询→JOIN，IN→EXISTS |
| 🔴 Senior | Partitioning | Range/List/Hash，分区裁剪 |
| 🔴 Senior | Materialized views | 预计算存储，定期刷新 |
| 🔴 Senior | Isolation levels | RU/RC/RR/S，脏读/不可重复读/幻读 |
| 🔴 Senior | Deadlock | 循环等待，固定顺序预防 |
| 🔴 Senior | Optimistic/Pessimistic | 版本号 vs FOR UPDATE |
| 🔴 Senior | Normalization | 规范化减冗余，反规范化提速 |
| 🔴 Senior | Sharding | Range/Hash/Directory，选好分片键 |
| 🔴 Senior | Read-write separation | 写主读从，处理复制延迟 |

